# 1. Raster Damage Assessment

This notebook demonstrates a basic flood damage assessment using raster data for both hazard and exposure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from damagescanner import DamageScanner

## Define input data

We use the Kampen (Netherlands) sample data:
- **Hazard**: Flood inundation map (GeoTIFF)
- **Exposure**: Land use map (GeoTIFF)
- **Vulnerability**: Depth-damage curves and maximum damage values

In [ ]:
data_path = Path("..") / "data" / "kampen"

hazard = data_path / "hazard" / "1in100_inundation_map.tif"
exposure = data_path / "exposure" / "landuse_map.tif"
curves = data_path / "vulnerability" / "curves.csv"
maxdam = data_path / "vulnerability" / "maxdam.csv"

## Initialize DamageScanner

The scanner automatically detects this is a raster-based assessment.

In [ ]:
ds = DamageScanner(
    hazard_data=hazard,
    feature_data=exposure,
    curves=curves,
    maxdam=maxdam,
)

print(f"Assessment type: {ds.assessment_type}")

## Calculate damages

Returns:
- `damage_df`: DataFrame with damages per land use class
- `damage_map`: 2D array of damages per pixel
- `landuse`: Land use array
- `hazard`: Hazard intensity array

In [ ]:
damage_df, damage_map, landuse, hazard_arr = ds.calculate()

In [ ]:
print(f"Total damage: €{damage_df['damage'].sum():,.0f}")
damage_df

## Visualize results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Damage map
im = axes[0].imshow(damage_map, cmap="Reds")
axes[0].set_title("Damage Map")
plt.colorbar(im, ax=axes[0], label="Damage (€)")

# Damage by land use
damage_df.reset_index().plot.bar(x="landuse", y="damage", ax=axes[1], legend=False)
axes[1].set_title("Damage by Land Use")
axes[1].set_ylabel("Damage (€)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()